<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 La Red de Pesca: ¿Cuánto Confiar en un Promedio? 🎣
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 03
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Estadistica%20analisis%20y%20representacion%20de%20datos/03%20-%20Inferencia%20Estadistica/Para%20Dummies/00_Intervalos_Confianza_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 00 del Módulo 03 (Intervalos de confianza)](../00_Intervalos_de_Confianza.ipynb). Imagina que eres comprador en la **plaza de mercado**: llega un camión con **2 000 bultos de papa** y la etiqueta dice «50 kg cada uno». No vas a pesar los 2 000. **Pesas 10 y sacas el promedio.** Pero otro comprador pesa otros 10 y obtiene **otro** promedio. ¿Cuánto puedes confiar en el tuyo?

| En la plaza | En estadística |
|---|---|
| Pesar los 2 000 bultos | Medir a **toda la población** (casi nunca se puede) |
| Pesar solo 10 | La **muestra** |
| El promedio de esos 10 | La **estimación** |
| «Entre 48.6 y 51.4 kg» | El **intervalo de confianza** |
| Una red de pesca que atrapa al pez el 95 % de las veces | El **«95 % de confianza»** |
| Volver a sacar bultos «de la bolsa» de los que ya pesaste | El **bootstrap** |

Al terminar podrás explicar:
1. Por qué **un solo promedio no basta** y qué es un **intervalo** (un promedio con su margen de error).
2. Qué significa realmente **«95 % de confianza»** (spoiler: **no** es lo que mucha gente cree).
3. Por qué pesar **más bultos** da un intervalo más angosto, y cómo se saca un intervalo **con una sola muestra pequeña** volviendo a sacar de la bolsa.

> 🔗 Si luego quieres ver las versiones **con fórmulas, comparación de métodos y simulaciones de cobertura**: [cuaderno estándar](../00_Intervalos_de_Confianza.ipynb). Taller de todo el módulo: [`../../homeworks/Para Dummies/03_Inferencia_Hands_On_Dummies.ipynb`](../../homeworks/Para%20Dummies/03_Inferencia_Hands_On_Dummies.ipynb).

---
## 1. Un Número Solo No Basta ⚖️

El camión es **inventado**, y por eso conocemos el peso de **todos** los bultos (en la vida real no). Así podemos comprobar si tu promedio se acerca a la verdad. Mira lo que pasa si pesamos **10 bultos, cinco veces distintas**, y luego cómo se arma un **intervalo**: *promedio ± margen de error*. El margen es más grande si los bultos se parecen poco entre sí y más chico si pesas muchos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

# Un camion con 2000 bultos de papa (INVENTADO). Aqui conocemos el peso de TODOS, asi que sabemos la verdad.
camion = rng.normal(50, 2, 2000).round(1)
promedio_real = camion.mean()
print(f"Promedio REAL de los 2000 bultos: {promedio_real:.2f} kg  (en la vida real esto NO se sabe)")

# Pesamos solo 10 bultos... y lo repetimos 5 veces con bultos distintos
promedios = []
for i in range(5):
    muestra = rng.choice(camion, 10, replace=False)
    promedios.append(muestra.mean())
    print(f"  Muestra {i + 1}: promedio = {muestra.mean():.2f} kg")
assert len(set(np.round(promedios, 6))) == 5 and abs(promedio_real - 50) < 0.3        # cada muestra da un numero distinto


def intervalo(muestra, factor=2.26):
    '''Promedio +/- margen de error. El 'factor' 2.26 es la regla para 10 bultos (no hay que memorizarla).'''
    margen = factor * muestra.std(ddof=1) / np.sqrt(len(muestra))
    return muestra.mean() - margen, muestra.mean() + margen


muestra = rng.choice(camion, 10, replace=False)
bajo, alto = intervalo(muestra)
print(f"\nMi muestra de 10 bultos: promedio = {muestra.mean():.2f} kg -> intervalo: entre {bajo:.2f} y {alto:.2f} kg")
assert bajo < muestra.mean() < alto and alto - bajo < 5


---
**Lección:** el peso real promedio del camión es **49.89 kg**, pero cada muestra de 10 dio un número **distinto** (48.57, 49.17, 49.37, 50.01, 49.83): **no es que alguien pesara mal**, es que cada grupo de 10 bultos es diferente. Por eso **un promedio solo es una «foto borrosa»**, y conviene decir «**entre 48.61 y 51.35 kg**» en lugar de «49.98 kg exactos». Eso es un **intervalo de confianza**.

---
## 2. La Red de Pesca: ¿Qué Significa «95 %»? 🎣

Un intervalo «del 95 %» es como una **red de pesca**: **no** significa «hay 95 % de probabilidad de que el pez esté dentro de *esta* red ya lanzada» (el pez está dentro o no). Significa que, si **mil vendedores** lanzan **su** red con el mismo método, **unos 950 atraparán** el promedio real y unos 50 no. Lo comprobamos, y luego vemos qué pasa al **pesar más bultos**.

In [ ]:
# 1 000 vendedores, cada uno pesa 10 bultos y calcula su intervalo. ¿Cuantos atrapan el promedio real?
atrapan = []
for _ in range(1000):
    m = rng.choice(camion, 10, replace=False)
    b, a = intervalo(m)
    atrapan.append(b <= promedio_real <= a)
porcentaje = np.mean(atrapan)
print(f"De 1 000 vendedores, {int(sum(atrapan))} atraparon el promedio real: {porcentaje:.1%}")
assert 0.92 < porcentaje < 0.98                                              # la 'red' funciona ~95 % de las veces

# Dibujamos los intervalos de los primeros 30 vendedores
fig, ax = plt.subplots(figsize=(8, 5))
rng2 = np.random.default_rng(1)
for i in range(30):
    m = rng2.choice(camion, 10, replace=False)
    b, a = intervalo(m)
    ok = b <= promedio_real <= a
    ax.plot([b, a], [i, i], color="#2563eb" if ok else "#dc2626", lw=2)
ax.axvline(promedio_real, color="black", ls="--", label="promedio REAL")
ax.set(xlabel="peso (kg)", ylabel="vendedor", title="30 vendedores, 30 intervalos (rojo = no atrapo la verdad)"); ax.legend()
plt.show()

# Mas bultos pesados = intervalo mas angosto (y cuadruplicar los bultos reduce el ancho a la MITAD)
reglas = {10: 2.26, 40: 2.02, 160: 1.98}                                       # factores para 10, 40 y 160 bultos
anchos = {}
for n_bultos, f in reglas.items():
    w = []
    for _ in range(500):
        b, a = intervalo(rng.choice(camion, n_bultos, replace=False), f)
        w.append(a - b)
    anchos[n_bultos] = np.mean(w)
    print(f"Con {n_bultos:3d} bultos, el intervalo mide en promedio {anchos[n_bultos]:.2f} kg de ancho")
assert 1.7 < anchos[10] / anchos[40] < 2.3 and 1.7 < anchos[40] / anchos[160] < 2.3

# Datos para la Practica 1: ingresos mensuales (millones de pesos) de 12 hogares INVENTADOS, con un par de hogares muy altos
hogares = np.array([1.2, 1.4, 1.5, 1.6, 1.8, 2.0, 2.1, 2.4, 2.9, 3.5, 6.8, 9.5])


---
**Lección:** de **1 000 vendedores**, **946** atraparon el promedio real (**94.6 %**, muy cerca del 95 %); los otros se quedaron «sin pez». Ninguno puede saber **si le tocó** el grupo de los que fallan: la confianza está **en el método**, no en cada intervalo. Y **más bultos, intervalo más angosto**: con 10 bultos el intervalo mide ≈ **2.80 kg**, con 40 ≈ **1.27 kg** y con 160 ≈ **0.63 kg**: **cuadruplicar** los bultos **reduce a la mitad** el ancho (pesar 4 veces más cuesta 4 veces más; la precisión solo se duplica). Un detalle de honestidad: este método supone que los pesos «se reparten más o menos en campana»; con datos muy torcidos y pocas observaciones puede fallar más de lo que promete (la práctica de abajo muestra una salida).

---
## Resumen en una frase 🎯

> **Un promedio de muestra es una estimación borrosa; el intervalo de confianza le pone un margen de error, «95 %» describe lo bien que funciona el método a la larga (no una probabilidad sobre un intervalo ya calculado), y más datos lo hacen más angosto, pero solo a un ritmo lento.**

### Lo que aprendiste hoy:
- ✅ **Cada muestra da un promedio distinto**: por eso se reporta un **intervalo**, no un solo número.
- ✅ **«95 %»** = la red de pesca atrapa al promedio real **~95 de cada 100 veces**; en la simulación fueron 946 de 1 000.
- ✅ **Más bultos, intervalo más angosto**, pero para reducirlo a la mitad hay que **cuadruplicar** la muestra.
- ✅ Con pocos datos y valores muy altos, se puede **volver a sacar de la bolsa** (bootstrap) para armar un intervalo sin fórmulas.

---
##### 🛠️ Práctica 1: Volver a sacar de la bolsa: el intervalo de la mediana

Con el arreglo `hogares` (ingresos mensuales, en millones de pesos, de 12 hogares **inventados**, dos de ellos muy altos) y la idea del **bootstrap**:

1. Crea un generador `rng_b = np.random.default_rng(7)` y repite **5 000 veces**: saca 12 hogares **con reemplazo** (`rng_b.choice(hogares, len(hogares), replace=True)`) y guarda su **mediana** en la lista `medianas`.
2. Calcula `bajo` y `alto` como los percentiles 2.5 y 97.5 de `medianas` (`np.percentile`). Ese es tu **intervalo del 95 %** para la mediana.
3. Verifica con `assert` que `bajo < np.median(hogares) < alto`. Comenta: ¿por qué la **mediana** representa mejor a estos hogares que el **promedio**?

In [ ]:
# Escribe tu código aquí

# rng_b = np.random.default_rng(7)
# medianas = []
# for _ in range(5000):
#     ...
# bajo, alto = ...


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
rng_b = np.random.default_rng(7)
medianas = []
for _ in range(5000):
    remuestra = rng_b.choice(hogares, len(hogares), replace=True)                  # sacar de la bolsa y devolver
    medianas.append(np.median(remuestra))
bajo, alto = np.percentile(medianas, [2.5, 97.5])
print(f"Mediana = {np.median(hogares):.2f} | promedio = {hogares.mean():.2f} | intervalo del 95 % para la mediana: entre {bajo:.2f} y {alto:.2f}")

assert bajo < np.median(hogares) < alto
print("Los dos hogares muy altos (6.8 y 9.5) estiran el promedio; la mediana describe mejor al hogar 'tipico'. Con 12 datos el intervalo es ancho: honestidad ante todo.")
```
</details>

---

---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿por qué cinco muestras de 10 bultos dieron cinco promedios distintos, si el camión es el mismo? ¿Qué le dirías a alguien que dice «pesaron mal»?
2. Un colega afirma: «mi intervalo del 95 % es 48.6–51.4, así que hay 95 % de probabilidad de que el promedio real esté ahí». ¿Qué corregirías usando la imagen de la red de pesca?
3. Si quieres que tu intervalo sea **la mitad de ancho**, ¿cuántos bultos más tendrías que pesar? ¿Vale la pena si cada bulto cuesta tiempo y dinero?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos (Edición Didáctica Para Dummies)</i>
  </p>
</div>